In [4]:
import openai

In [9]:
import os

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [17]:
from llama_index import VectorStoreIndex
from llama_index.schema import Document
import requests

article = requests.get("https://www.technologyreview.com/2021/12/06/1041345/ai-nlp-mental-health-better-therapists-psychology-cbt/").text


In [34]:
doc = Document(text=article, batch_size=128)

In [19]:
index = VectorStoreIndex.from_documents([doc])

In [20]:
query_engine = index.as_query_engine()


In [21]:
response = query_engine.query("Explain the value of AI in three sentences accroding to the article.")

In [22]:
print(response)

AI has had a significant impact on various industries, including finance and therapy. It has enabled financial services firms to adopt generative AI, potentially generating income from the technology. In the field of therapy, AI is being used to analyze the language therapists use with clients, leading to a better understanding of how therapy works and potentially improving outcomes for patients.


In [25]:
import chromadb

In [28]:
client = chromadb.PersistentClient()

collection = client.create_collection("test")

In [42]:
from llama_index.embeddings import LangchainEmbedding
from llama_index import StorageContext, ServiceContext
from llama_index.vector_stores import ChromaVectorStore
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
embed_model = LangchainEmbedding(HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2'))
service_context_embedding = ServiceContext.from_defaults(embed_model=embed_model, chunk_size=128, chunk_overlap=15)

In [43]:
doc = Document(text=article)

index = VectorStoreIndex.from_documents([doc], storage_context=storage_context, service_context=service_context_embedding, insert_batch_size=128)

In [59]:
from llama_index.query_engine import RetrieverQueryEngine
from llama_index.llms import OpenAI

service_context_llm = ServiceContext.from_defaults(
    llm=OpenAI(
        model="text-davinci-002",
        temperature=0.1,
    ),
    system_prompt="You are an AI assistant answering questions related to websites."
)

query_engine = index.as_query_engine(service_context=service_context_llm)

In [60]:
response = query_engine.query("Explain the value of AI in three sentences accroding to the article.")

NotFoundError: Error code: 404 - {'error': {'message': 'The model `text-davinci-002` has been deprecated, learn more here: https://platform.openai.com/docs/deprecations', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [49]:
print(query_engine.query("Explain soccer to me"))

I'm sorry, but the given context information does not provide any information about soccer. Could you please provide more specific details or ask a different question?


In [51]:
print(query_engine.query("Explain the value of AI in three sentences accroding to the article."))

AI has the potential to improve human care by providing additional support and resources. It is not meant to replace human care, but rather enhance it. The value of AI lies in its ability to address the lack of quality mental-health care by reducing stigma, increasing funding, and improving education.
